In [18]:

import json
from typing import Dict, List, Any, Literal
from load_json import load_json_file

paths:list[str] = [
    # '../data/western_cookies.json',
    '../data/indonesian_baked_goods.json',
    '../data/indonesian_drinks.json'
]

inspiring_set:list[Dict[Unknown, Unknown]] = []

for path in paths:
    data:list[Dict[Unknown, Unknown]] = load_json_file(path)
    inspiring_set.extend(data)

print(str(inspiring_set))

[{'name': 'Es Kelapa Muda (Indofood)', 'family': 'drink', 'servings': 1, 'source': 'https://arc.net/l/quote/zikegnul', 'ingredients': [{'name': 'young coconut water', 'amount': 240, 'unit': 'ml', 'flavour': 'sweet', 'type': 'Beverages', 'cons': 'LIQUID'}, {'name': 'young coconut flesh', 'amount': 75, 'unit': 'g', 'flavour': 'sweet', 'type': 'Produce', 'cons': 'SOLID'}, {'name': 'sugar syrup', 'amount': 15, 'unit': 'ml', 'flavour': 'sweet', 'type': 'Baking', 'cons': 'LIQUID'}, {'name': 'ice', 'amount': 100, 'unit': 'g', 'flavour': 'none', 'type': 'Frozen', 'cons': 'SOLID'}, {'name': 'pandan leaf', 'amount': 5, 'unit': 'g', 'flavour': 'aromatic', 'type': 'Spices and Seasonings', 'cons': 'SOLID'}]}, {'name': 'Es Alpukat', 'family': 'drink', 'servings': 1, 'source': 'https://www.indofoodstore.com/blogs/news/es-alpukat-recipe-indonesian-ingredients', 'ingredients': [{'name': 'avocado', 'amount': 150, 'unit': 'g', 'flavour': 'creamy', 'type': 'Produce', 'cons': 'SOLID'}, {'name': 'sweetened 

In [19]:
def get_unique_values(key:str) -> set:
    return {
        ingredient[key]
        for recipe in inspiring_set
        for ingredient in recipe["ingredients"]
    }

unique_flavours = get_unique_values("flavour")
unique_types = get_unique_values("type")
unique_ingredients = get_unique_values("name")
unique_families = {recipe["family"] for recipe in inspiring_set}

print(unique_flavours)
print(unique_types)
print(unique_ingredients)
print(unique_families)

{'acidity', 'salt', 'spice', 'bitter', 'creamy', 'sweet', 'none', 'aromatic'}
{'Beverages', 'Pasta and Rice', 'Cheese', 'Milk, Eggs, Other Dairy', 'Produce', 'Oil, Vinegar, Salad Dressing', 'Baking', 'Health Foods', 'Canned and Jarred', 'Nuts', 'Frozen', 'Nut butters, Jams, and Honey', 'Spices and Seasonings', 'Ethnic Foods', 'Tea and Coffee', 'Bread'}
{'grated cheese', 'water', 'sugar palm fruit', 'baking soda', 'cumin', 'pandan essence', 'kaffir lime leaf', 'banana leaf', 'nutmeg', 'soybeans', 'basil seed', 'tamarind', 'honey', 'turmeric powder', 'white peppercorn', 'vanilla powder', 'young coconut water', 'sweetened condensed milk', 'food colouring', 'bay leaf', 'cinnamon', 'evaporated milk', 'soda water', 'fermented cassava (tape)', 'butter', 'black tea', 'cloves', 'vanilla sugar', 'ground coffee', 'cocopandan syrup', 'sinom leaf', 'star anise', 'wheat flour', 'mung bean starch', 'white rice', 'kepok banana', 'pandan leaf', 'plain flour', 'cardamom', 'sugar', 'melon', 'young coconu

In [ ]:
%pip install scipy
from typing import TypedDict
import pprint
import numpy as np
from scipy import stats
recipe_flavour_ratios = []

# def get_ingredient_ratios(recipe: Recipe, key: str) -> float:
#     total_recipe_amount = sum(ing["amount"] for ing in recipe["ingredients"])
#     amounts_per_category = {}
#     for ingredient in recipe["ingredients"]:
#         category = ingredient[key]
#         amounts_per_category[category] =+ ingredient["amount"]

#     for category, amount in amounts_per_category.items():
#         amounts_per_category[category] = amount / total_recipe_amount if total_recipe_amount > 0 else 0

#     return amounts_per_category

def get_ingredient_ratios(recipe: Dict[str, Any], key: str) -> Dict[str, float]:
    total_recipe_amount = sum(
        ingredient["amount"] for ingredient in recipe["ingredients"]
    )

    amounts_per_category: Dict[str, float] = {}

    for ingredient in recipe["ingredients"]:
        category = ingredient[key]
        amounts_per_category[category] = (
            amounts_per_category.get(category, 0) + ingredient["amount"]
        )

    return {
        category: amount / total_recipe_amount if total_recipe_amount > 0 else 0
        for category, amount in amounts_per_category.items()
    }

class RatioResult(TypedDict):
    name: str
    ratios: dict[str, float]


class CategoryStatistics(TypedDict):
    mean: float
    min: float
    max: float


def calculate_category_statistics(
    recipes: list[dict[str, Any]],
    category_key: str,
) -> tuple[list[RatioResult], dict[str, CategoryStatistics]]:
    categories: set[str] = {
        str(ingredient[category_key])
        for recipe in recipes
        for ingredient in recipe["ingredients"]
    }

    recipe_ratios: list[RatioResult] = []

    for recipe in recipes:
        ratios = get_ingredient_ratios(recipe, category_key)

        complete_ratios = {
            category: ratios.get(category, 0.0)
            for category in categories
        }

        recipe_ratios.append({
            "name": str(recipe["name"]),
            "ratios": complete_ratios,
        })

    statistics: dict[str, CategoryStatistics] = {
        category: {
            "mean": sum(
                item["ratios"][category] for item in recipe_ratios
            ) / len(recipe_ratios),
            "min": min(item["ratios"][category] for item in recipe_ratios),
            "max": max(item["ratios"][category] for item in recipe_ratios),
        }
        for category in categories
    }

    return recipe_ratios, statistics


recipe_flavour_ratios, flavour_statistics = calculate_category_statistics(
    inspiring_set,
    "flavour",
)

recipe_type_ratios, type_statistics = calculate_category_statistics(
    inspiring_set,
    "type",
)

print("Flavour Statistics:", pprint.pformat(flavour_statistics))
print("Type Statistics:", pprint.pformat(type_statistics))

Flavour Statistics: {'acidity': {'max': 0.22371364653243847,
             'mean': 0.012610533234090978,
             'min': 0.0},
 'aromatic': {'max': 0.38871951219512196,
              'mean': 0.020651303838833082,
              'min': 0.0},
 'bitter': {'max': 0.14109347442680775,
            'mean': 0.006801153718857514,
            'min': 0.0},
 'creamy': {'max': 0.7537688442211056, 'mean': 0.15454724806392225, 'min': 0.0},
 'none': {'max': 0.9927007299270073, 'mean': 0.5872045306309784, 'min': 0.0},
 'salt': {'max': 0.005277044854881266,
          'mean': 0.0010696354325280442,
          'min': 0.0},
 'spice': {'max': 0.12, 'mean': 0.017011286527471157, 'min': 0.0},
 'sweet': {'max': 0.7586206896551724, 'mean': 0.20010430855331854, 'min': 0.0}}
Type Statistics: {'Baking': {'max': 0.8158844765342961, 'mean': 0.13494373288045716, 'min': 0.0},
 'Beverages': {'max': 0.9523809523809523,
               'mean': 0.39559909797524245,
               'min': 0.0},
 'Bread': {'max': 0.04, 'mean